# Analisis de embeddings latentes Aβ/SPECT

Este notebook analiza los embeddings generados durante la fase 1 del entrenamiento, guardados en `data/latent_Abeta`.

Objetivos:
- cargar los latentes `.nii` / `.nii.gz`;
- asociarlos con la clase en `data_info/data_info.csv`;
- revisar balance de clases y disponibilidad de archivos;
- proyectar los embeddings con PCA, t-SNE y UMAP;
- visualizar cada metodo en 2D y 3D;
- calcular metricas exploratorias simples de separacion entre clases;
- comparar, desde la línea base, la distribución de los embeddings SPECT y MRI de cada autoencoder nuevo (sección 13).

Suposicion de etiquetas: `0 = control`, `1 = parkinson`. Si el mapeo real es distinto, cambia `LABEL_NAMES` en la celda de configuracion.

## 0. Dependencias opcionales

Ejecuta esta celda si el kernel no tiene las librerias de visualizacion/reduccion dimensional. En el contenedor base detectamos que pueden faltar `scikit-learn`, `umap-learn`, `plotly` y `matplotlib`.

In [ ]:
# Descomenta si hacen falta dependencias en el kernel de Jupyter.
%pip install scikit-learn umap-learn matplotlib plotly seaborn

In [ ]:
import numpy as np
import os
import matplotlib.pyplot as plt

In [ ]:
from pathlib import Path

def find_repo_root() -> Path:
    p = Path.cwd().resolve()
    for parent in [p, *p.parents]:
        if (parent / "config.py").is_file():
            return parent
    raise FileNotFoundError("Could not find repo root (config.py)")

PROJECT_ROOT = find_repo_root()
INFER_RUN = PROJECT_ROOT / "result/exp_1/inference/20260508_160032/representations"
#file = INFER_RUN / "train/3102.npz"
#data = np.load(file)
#synthetic_spect_flat = data["sampled_latent_final"]

#print(type(synthetic_spect_flat))
#print(synthetic_spect_flat.shape)
#print(synthetic_spect_flat.dtype)

In [ ]:
volume = synthetic_spect_flat[0]
slice_idx = 20

plt.figure(figsize=(5, 5))
plt.imshow(volume[slice_idx, :, :], cmap="gray")
plt.colorbar()
plt.title(f"sampled_latent_final - slice {slice_idx}")
plt.axis("off")
plt.show()

In [ ]:
len(os.listdir(INFER_RUN / "test"))

## 1. Configuracion e imports

In [ ]:
from pathlib import Path
import warnings

import nibabel as nib
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
try:
    import nbformat  # required by Plotly inside notebooks
    import plotly.express as px
    HAS_PLOTLY = True
except ImportError:
    px = None
    HAS_PLOTLY = False

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score

try:
    import umap
except ImportError as exc:
    raise ImportError('Falta umap-learn. Ejecuta: %pip install umap-learn') from exc

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')

def find_repo_root() -> Path:
    p = Path.cwd().resolve()
    for parent in [p, *p.parents]:
        if (parent / "config.py").is_file():
            return parent
    raise FileNotFoundError("Could not find repo root (config.py)")

PROJECT_ROOT = find_repo_root()
LATENT_DIR = PROJECT_ROOT / 'data' / 'latent_Abeta'
DATA_INFO = PROJECT_ROOT / 'data_info' / 'data_info.csv'

LABEL_NAMES = {
    0: 'control',
    1: 'parkinson',
}

RANDOM_STATE = 42
PCA_PREPROCESS_COMPONENTS = 50
TSNE_PERPLEXITY = 30
UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.1

print('Latent dir:', LATENT_DIR)
print('Data info:', DATA_INFO)

## 2. Cargar metadata y latentes

Cada archivo en `data/latent_Abeta` se identifica por el nombre del archivo, por ejemplo `3102.nii.gz -> ID=3102`. Solo se usan latentes que tengan etiqueta en `data_info.csv`.

In [ ]:
def latent_id(path: Path) -> str:
    name = path.name
    if name.endswith('.nii.gz'):
        return name[:-7]
    return path.stem


def load_latent(path: Path) -> np.ndarray:
    data = nib.load(str(path)).get_fdata(dtype=np.float32)
    return np.asarray(data, dtype=np.float32)


info = pd.read_csv(DATA_INFO, dtype={'ID': str})
info['label'] = info['label'].astype(int)
info['class_name'] = info['label'].map(LABEL_NAMES).fillna(info['label'].astype(str))

latent_paths = sorted(list(LATENT_DIR.glob('*.nii')) + list(LATENT_DIR.glob('*.nii.gz')))
latent_df = pd.DataFrame({
    'ID': [latent_id(path) for path in latent_paths],
    'latent_path': latent_paths,
})

merged = latent_df.merge(info, on='ID', how='left', indicator=True)
missing_labels = merged[merged['_merge'] != 'both']
if len(missing_labels):
    print('Latentes sin etiqueta en data_info.csv:')
    display(missing_labels[['ID', 'latent_path']])

merged = merged[merged['_merge'] == 'both'].drop(columns=['_merge']).reset_index(drop=True)
display(merged.head())
print(f'Latentes encontrados: {len(latent_paths)}')
print(f'Latentes con etiqueta: {len(merged)}')

In [ ]:
merged.groupby('label').size()

## 3. Control de calidad inicial

In [ ]:
print('Distribucion por clase:')
display(merged.groupby(['label', 'class_name']).size().rename('n').reset_index())

if 'split' in merged.columns:
    print('Distribucion por split y clase:')
    display(merged.groupby(['split', 'class_name']).size().rename('n').reset_index())

sample_shapes = []
for path in merged['latent_path'].head(min(10, len(merged))):
    sample_shapes.append(load_latent(path).shape)
print('Shapes observados en muestra:', sorted(set(sample_shapes)))

## 4. Construir matriz de embeddings

Se aplana cada volumen latente 3D a un vector. Para t-SNE y UMAP se usa una reduccion PCA previa a maximo 50 componentes, una practica comun para reducir ruido y costo computacional.

In [ ]:
latents = []
shapes = []
for path in merged['latent_path']:
    volume = load_latent(path)
    shapes.append(volume.shape)
    latents.append(volume.reshape(-1))

if len(set(shapes)) != 1:
    raise ValueError(f'Los latentes no tienen todos el mismo shape: {sorted(set(shapes))}')

X = np.stack(latents).astype(np.float32)
y = merged['label'].to_numpy()
class_names = merged['class_name'].to_numpy()

print('Shape latente 3D:', shapes[0])
print('Matriz X:', X.shape)
print('Memoria aproximada X:', f'{X.nbytes / 1024**2:.1f} MB')

scaler = StandardScaler(with_mean=True, with_std=True)
X_scaled = scaler.fit_transform(X)

pre_components = min(PCA_PREPROCESS_COMPONENTS, X_scaled.shape[0] - 1, X_scaled.shape[1])
pca_pre = PCA(n_components=pre_components, random_state=RANDOM_STATE)
X_pre = pca_pre.fit_transform(X_scaled)

print('X_pre para t-SNE/UMAP:', X_pre.shape)
print('Varianza explicada PCA previa:', f'{pca_pre.explained_variance_ratio_.sum():.3f}')

## 5. Funciones de visualizacion y metricas

In [ ]:
def make_projection_df(coords, method):
    cols = [f'{method}_{i+1}' for i in range(coords.shape[1])]
    df = pd.DataFrame(coords, columns=cols)
    df['ID'] = merged['ID'].values
    df['label'] = y
    df['class_name'] = class_names
    if 'split' in merged.columns:
        df['split'] = merged['split'].values
    return df


def plot_2d(df, method):
    x_col, y_col = f'{method}_1', f'{method}_2'
    plt.figure(figsize=(8, 6))
    sns.scatterplot(
        data=df,
        x=x_col,
        y=y_col,
        hue='class_name',
        style='split' if 'split' in df.columns else None,
        s=70,
        alpha=0.85,
    )
    plt.title(f'{method} 2D de embeddings latentes')
    plt.xlabel(x_col)
    plt.ylabel(y_col)
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()


def plot_3d(df, method):
    x_col, y_col, z_col = f'{method}_1', f'{method}_2', f'{method}_3'
    if HAS_PLOTLY:
        fig = px.scatter_3d(
            df,
            x=x_col,
            y=y_col,
            z=z_col,
            color='class_name',
            symbol='split' if 'split' in df.columns else None,
            hover_data=['ID', 'label'] + (['split'] if 'split' in df.columns else []),
            title=f'{method} 3D de embeddings latentes',
            width=850,
            height=650,
        )
        fig.show()
        return

    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')
    for class_name, group in df.groupby('class_name'):
        ax.scatter(
            group[x_col],
            group[y_col],
            group[z_col],
            label=class_name,
            s=45,
            alpha=0.8,
        )
    ax.set_title(f'{method} 3D de embeddings latentes')
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_zlabel(z_col)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()


def projection_metrics(coords, labels, method):
    if len(np.unique(labels)) < 2 or len(labels) < 3:
        return {'method': method, 'silhouette': np.nan, 'davies_bouldin': np.nan}
    return {
        'method': method,
        'silhouette': silhouette_score(coords, labels),
        'davies_bouldin': davies_bouldin_score(coords, labels),
    }

## 6. PCA 2D y 3D

In [ ]:
pca_2d = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X_scaled)
pca_3d = PCA(n_components=3, random_state=RANDOM_STATE).fit_transform(X_scaled)

pca_df_2d = make_projection_df(pca_2d, 'PCA')
pca_df_3d = make_projection_df(pca_3d, 'PCA')

print('Varianza explicada PCA 2D:', PCA(n_components=2, random_state=RANDOM_STATE).fit(X_scaled).explained_variance_ratio_.sum())
print('Varianza explicada PCA 3D:', PCA(n_components=3, random_state=RANDOM_STATE).fit(X_scaled).explained_variance_ratio_.sum())

plot_2d(pca_df_2d, 'PCA')
plot_3d(pca_df_3d, 'PCA')

## 7. t-SNE 2D y 3D

`perplexity` debe ser menor que el numero de muestras. Si hay pocos sujetos, se ajusta automaticamente.

In [ ]:
perplexity = min(TSNE_PERPLEXITY, max(5, (len(X_pre) - 1) // 3))
print('Perplexity usada:', perplexity)

tsne_2d = TSNE(
    n_components=2,
    perplexity=perplexity,
    init='pca',
    learning_rate='auto',
    random_state=RANDOM_STATE,
).fit_transform(X_pre)

tsne_3d = TSNE(
    n_components=3,
    perplexity=perplexity,
    init='pca',
    learning_rate='auto',
    random_state=RANDOM_STATE,
).fit_transform(X_pre)

tsne_df_2d = make_projection_df(tsne_2d, 'TSNE')
tsne_df_3d = make_projection_df(tsne_3d, 'TSNE')

plot_2d(tsne_df_2d, 'TSNE')
plot_3d(tsne_df_3d, 'TSNE')

## 8. UMAP 2D y 3D

In [ ]:
umap_2d = umap.UMAP(
    n_components=2,
    n_neighbors=UMAP_N_NEIGHBORS,
    min_dist=UMAP_MIN_DIST,
    metric='euclidean',
    random_state=RANDOM_STATE,
).fit_transform(X_pre)

umap_3d = umap.UMAP(
    n_components=3,
    n_neighbors=UMAP_N_NEIGHBORS,
    min_dist=UMAP_MIN_DIST,
    metric='euclidean',
    random_state=RANDOM_STATE,
).fit_transform(X_pre)

umap_df_2d = make_projection_df(umap_2d, 'UMAP')
umap_df_3d = make_projection_df(umap_3d, 'UMAP')

plot_2d(umap_df_2d, 'UMAP')
plot_3d(umap_df_3d, 'UMAP')

## 9. Comparacion cuantitativa exploratoria

Estas metricas no sustituyen validacion clinica. Sirven para estimar si las clases forman agrupamientos en cada proyeccion.

- `silhouette`: mayor es mejor, rango aproximado [-1, 1].
- `davies_bouldin`: menor es mejor.

In [ ]:
metrics = pd.DataFrame([
    projection_metrics(pca_2d, y, 'PCA_2D'),
    projection_metrics(pca_3d, y, 'PCA_3D'),
    projection_metrics(tsne_2d, y, 'TSNE_2D'),
    projection_metrics(tsne_3d, y, 'TSNE_3D'),
    projection_metrics(umap_2d, y, 'UMAP_2D'),
    projection_metrics(umap_3d, y, 'UMAP_3D'),
])
display(metrics.sort_values('silhouette', ascending=False))

## 10. Guardar proyecciones para analisis posterior

In [ ]:
OUT_DIR = PROJECT_ROOT / 'result' / 'embedding_analysis'
OUT_DIR.mkdir(parents=True, exist_ok=True)

pca_df_2d.to_csv(OUT_DIR / 'pca_2d.csv', index=False)
pca_df_3d.to_csv(OUT_DIR / 'pca_3d.csv', index=False)
tsne_df_2d.to_csv(OUT_DIR / 'tsne_2d.csv', index=False)
tsne_df_3d.to_csv(OUT_DIR / 'tsne_3d.csv', index=False)
umap_df_2d.to_csv(OUT_DIR / 'umap_2d.csv', index=False)
umap_df_3d.to_csv(OUT_DIR / 'umap_3d.csv', index=False)
metrics.to_csv(OUT_DIR / 'projection_metrics.csv', index=False)

print('Resultados guardados en:', OUT_DIR)

## 11. Clasificacion supervisada de embeddings

En esta seccion se evalua si los embeddings latentes originales contienen informacion discriminativa para separar `control` vs `parkinson`.

Como los embeddings disponibles se tratan aqui como un unico conjunto de entrenamiento, se crea una particion interna estratificada `80/20` para entrenamiento y prueba.

**Importante:** esta clasificacion usa los embeddings en su version original aplanada. No se aplica PCA, t-SNE, UMAP ni ninguna tecnica de reduccion dimensional. Solo se estandarizan las features con `StandardScaler`, ajustado exclusivamente sobre train.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
)

X_train_raw, X_test_raw, y_train, y_test, meta_train, meta_test = train_test_split(
    X,
    y,
    merged,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("Train:", X_train_raw.shape, "Test:", X_test_raw.shape)
print("Features originales por embedding:", X_train_raw.shape[1])
print("Distribucion train:", pd.Series(y_train).map(LABEL_NAMES).value_counts().to_dict())
print("Distribucion test:", pd.Series(y_test).map(LABEL_NAMES).value_counts().to_dict())

### 11.1 Modelos clasicos de ML

Cada modelo usa el mismo preprocesamiento: `StandardScaler -> clasificador`. No hay reduccion dimensional. El scaler se ajusta solo con train.

Para KNN se reporta explicitamente el valor de `K` usado en la columna `n_neighbors`. Se incluye un KNN automatico, que usa el mayor impar hasta 5 permitido por el tamano de train, y un KNN adicional con `K=2` por el conocimiento previo de que existen dos clases.

In [ ]:
def build_pipeline(classifier):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", classifier),
    ])


knn_auto_neighbors = min(5, len(y_train))
if knn_auto_neighbors % 2 == 0:
    knn_auto_neighbors -= 1
knn_auto_neighbors = max(1, knn_auto_neighbors)
knn_k2_neighbors = min(2, len(y_train))

print(f"KNN automatico entrenado con K={knn_auto_neighbors}")
print(f"KNN adicional entrenado con K={knn_k2_neighbors}")

classical_models = {
    "KNN_auto": build_pipeline(KNeighborsClassifier(n_neighbors=knn_auto_neighbors, weights="distance")),
    "KNN_K2": build_pipeline(KNeighborsClassifier(n_neighbors=knn_k2_neighbors, weights="distance")),
    "SVM_RBF": build_pipeline(SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, class_weight="balanced", random_state=RANDOM_STATE)),
    "SVM_linear": build_pipeline(SVC(kernel="linear", C=1.0, probability=True, class_weight="balanced", random_state=RANDOM_STATE)),
    "LogisticRegression": build_pipeline(LogisticRegression(max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE)),
}


def positive_scores(model, X_eval):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X_eval)[:, 1]
    if hasattr(model, "decision_function"):
        scores = model.decision_function(X_eval)
        return (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)
    return None


def classifier_params(model):
    classifier = model.named_steps["classifier"]
    params = {"classifier": classifier.__class__.__name__}
    if hasattr(classifier, "n_neighbors"):
        params["n_neighbors"] = classifier.n_neighbors
    if hasattr(classifier, "kernel"):
        params["kernel"] = classifier.kernel
    if hasattr(classifier, "C"):
        params["C"] = classifier.C
    return params


results = []
predictions = {}
for name, model in classical_models.items():
    model.fit(X_train_raw, y_train)
    y_pred = model.predict(X_test_raw)
    y_score = positive_scores(model, X_test_raw)
    auc = roc_auc_score(y_test, y_score) if y_score is not None and len(np.unique(y_test)) == 2 else np.nan
    row = {
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": auc,
    }
    row.update(classifier_params(model))
    results.append(row)
    predictions[name] = y_pred

classical_results = pd.DataFrame(results).sort_values("balanced_accuracy", ascending=False)
display(classical_results)

### 11.2 DNN sencilla

La DNN tambien usa los embeddings originales aplanados. Solo se aplica `StandardScaler` ajustado con train. Es intencionalmente pequena y regularizada para reducir sobreajuste, porque el numero de sujetos es bajo frente al numero de features.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

dnn_scaler = StandardScaler()
X_train_scaled = dnn_scaler.fit_transform(X_train_raw)
X_test_scaled = dnn_scaler.transform(X_test_raw)

X_train_dnn = X_train_scaled.astype(np.float32)
X_test_dnn = X_test_scaled.astype(np.float32)

class EmbeddingMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        hidden = 128
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.35),
            nn.Linear(hidden, 32),
            nn.ReLU(),
            nn.Dropout(0.35),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EmbeddingMLP(X_train_dnn.shape[1]).to(device)

class_counts = np.bincount(y_train.astype(int), minlength=2)
pos_weight = torch.tensor([class_counts[0] / max(class_counts[1], 1)], dtype=torch.float32, device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)

train_ds = TensorDataset(
    torch.tensor(X_train_dnn, dtype=torch.float32),
    torch.tensor(y_train.astype(np.float32), dtype=torch.float32),
)
train_loader = DataLoader(train_ds, batch_size=min(16, len(train_ds)), shuffle=True)

best_state = None
best_loss = float("inf")
patience = 30
bad_epochs = 0
history = []

X_train_tensor = torch.tensor(X_train_dnn, dtype=torch.float32, device=device)
X_test_tensor = torch.tensor(X_test_dnn, dtype=torch.float32, device=device)
y_train_tensor = torch.tensor(y_train.astype(np.float32), dtype=torch.float32, device=device)
y_test_tensor = torch.tensor(y_test.astype(np.float32), dtype=torch.float32, device=device)

for epoch in range(300):
    model.train()
    epoch_losses = []
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        epoch_losses.append(loss.item())

    model.eval()
    with torch.no_grad():
        train_logits = model(X_train_tensor)
        test_logits = model(X_test_tensor)
        test_loss = criterion(test_logits, y_test_tensor).item()

        train_preds = (torch.sigmoid(train_logits) >= 0.5).cpu().numpy().astype(int)
        test_preds = (torch.sigmoid(test_logits) >= 0.5).cpu().numpy().astype(int)

    train_loss = float(np.mean(epoch_losses))
    train_accuracy = accuracy_score(y_train, train_preds)
    test_accuracy = accuracy_score(y_test, test_preds)

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "test_loss": test_loss,
        "train_accuracy": train_accuracy,
        "test_accuracy": test_accuracy,
    })

    if test_loss < best_loss:
        best_loss = test_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        bad_epochs = 0
    else:
        bad_epochs += 1
        if bad_epochs >= patience:
            break

if best_state is not None:
    model.load_state_dict(best_state)

history_df = pd.DataFrame(history)
display(history_df.tail())

plt.figure(figsize=(7, 4))
plt.plot(history_df["epoch"], history_df["train_loss"], label="train")
plt.plot(history_df["epoch"], history_df["test_loss"], label="test")
plt.xlabel("epoch")
plt.ylabel("BCE loss")
plt.title("Curva de entrenamiento DNN")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(history_df["epoch"], history_df["train_accuracy"], label="train_accuracy")
plt.plot(history_df["epoch"], history_df["test_accuracy"], label="test_accuracy")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.title("Curva de accuracy DNN")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(X_test_tensor)
    dnn_scores = torch.sigmoid(logits).detach().cpu().numpy()
dnn_pred = (dnn_scores >= 0.5).astype(int)

dnn_result = pd.DataFrame([{
    "model": "DNN_simple",
    "accuracy": accuracy_score(y_test, dnn_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, dnn_pred),
    "f1": f1_score(y_test, dnn_pred),
    "roc_auc": roc_auc_score(y_test, dnn_scores) if len(np.unique(y_test)) == 2 else np.nan,
}])

all_classification_results = pd.concat([classical_results, dnn_result], ignore_index=True)
display(all_classification_results.sort_values("balanced_accuracy", ascending=False))

print("Reporte DNN:")
print(classification_report(y_test, dnn_pred, target_names=[LABEL_NAMES[0], LABEL_NAMES[1]]))

cm = confusion_matrix(y_test, dnn_pred)
plt.figure(figsize=(4, 3))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=[LABEL_NAMES[0], LABEL_NAMES[1]], yticklabels=[LABEL_NAMES[0], LABEL_NAMES[1]])
plt.xlabel("Prediccion")
plt.ylabel("Real")
plt.title("Matriz de confusion DNN")
plt.tight_layout()
plt.show()

### 11.3 Guardar resultados de clasificacion

## 13. Distribuciones del embedding, de la línea base en adelante

PASTA mantiene la imagen en un intervalo acotado y la difusión predice el dato limpio dentro de ese intervalo. Aquí el análogo es el código del autoencoder: lo que la difusión recorre tiene que caber en el clamp y acercarse a `N(0, 1)`.

`eps_x0` y `x0` no entran. Reutilizan el embedding de la línea base. Cada autoencoder nuevo se agrega al final de `EMBEDDING_STAGES`.

- **Almacenado:** los vóxeles tal como quedaron en disco.
- **Vista difusión (solo SPECT):** si la etapa tiene calibración, es el z-score global que stage 2 aplica antes de difundir. Si no, coincide con lo almacenado. El clamp actúa sobre esta vista. Su varianza es el MSE de predecir la media.
- La MRI es la condición, no el blanco de la difusión. Se compara el código almacenado con `N(0, 1)`, que es el blanco de la KL.

Si una carpeta no tiene volúmenes, la celda imprime `sin embeddings` y sigue. La MRI de minmax+KL está en ese caso: ese entrenamiento se lanzó con `--skip-embeddings`.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import norm


def find_repo_root() -> Path:
    p = Path.cwd().resolve()
    for parent in [p, *p.parents]:
        if (parent / "config.py").is_file():
            return parent
    raise FileNotFoundError("Could not find repo root (config.py)")


ROOT = find_repo_root()

# Orden cronológico. No reordenar: las figuras siguen esta secuencia.
EMBEDDING_STAGES = [
    {
        "key": "baseline",
        "title": "0 · Línea base",
        "change": "Encoder sin cota. Stage 2 hace z-score del SPECT; la cola queda.",
        "clamp": 10.0,
        "latent_calibration": True,
        "modalities": {
            "SPECT": ROOT / "data_rois/latent_Abeta_reproc",
            "MRI": ROOT / "result/exp_nm_s1_mri_rois_reproc/phase1_mri/embeddings/train",
        },
    },
    {
        "key": "minmax_kl",
        "title": "1 · minmax + KL",
        "change": "Imagen a [0,1] con percentiles fijos, KL hacia N(0,1), decoder tanh. SPECT sin z-score, clamp ±4.",
        "clamp": 4.0,
        "latent_calibration": False,
        "modalities": {
            "SPECT": ROOT / "data_rois/latent_Abeta_minmax_kl",
            "MRI": ROOT / "result/exp_nm_s1_mri_rois_reproc_minmax_kl/phase1_mri/embeddings/train",
        },
    },
]

STAGE_COLORS = {
    "baseline": "#4C78A8",
    "minmax_kl": "#54A24B",
}
PLOT_SAMPLE = 150_000
QQ_SAMPLE = 20_000
HIST_RANGE = (-12.0, 12.0)
RNG = np.random.default_rng(0)


def latent_files(root: Path) -> list[Path]:
    if not root.is_dir():
        return []
    files = []
    for pattern in ("*.nii", "*.nii.gz", "*.npy"):
        files.extend(root.rglob(pattern))
    return sorted(files)


def load_values(path: Path) -> np.ndarray:
    if path.suffix == ".npy":
        array = np.load(path)
    else:
        array = nib.load(str(path)).get_fdata(dtype=np.float32)
    return np.asarray(array, dtype=np.float64).ravel()


def load_directory(root: Path) -> np.ndarray | None:
    files = latent_files(root)
    if not files:
        return None
    return np.concatenate([load_values(path) for path in files])


def sample_values(values: np.ndarray, size: int) -> np.ndarray:
    if values.size <= size:
        return values
    return RNG.choice(values, size=size, replace=False)


def summarize(values: np.ndarray, clamp: float) -> dict:
    std = float(values.std())
    absolute = np.abs(values)
    percentiles = {
        name: float(np.percentile(values, pct))
        for name, pct in (("p0.1", 0.1), ("p1", 1.0), ("p50", 50.0), ("p99", 99.0), ("p99.9", 99.9))
    }
    return {
        "n_voxels": int(values.size),
        "mean": float(values.mean()),
        "std": std,
        "variance": float(std * std),
        "min": float(values.min()),
        "max": float(values.max()),
        **percentiles,
        "frac_|z|>1": float(np.mean(absolute > 1.0)),
        "frac_|z|>4": float(np.mean(absolute > 4.0)),
        "frac_fuera_clamp": float(np.mean(absolute > clamp)),
    }


def diffusion_view(values: np.ndarray, calibrate: bool) -> np.ndarray:
    if not calibrate:
        return values
    std = float(values.std())
    if std == 0.0:
        return values - float(values.mean())
    return (values - float(values.mean())) / std


rows = []
loaded = {}
for stage in EMBEDDING_STAGES:
    calibrate = bool(stage["latent_calibration"])
    clamp = float(stage["clamp"])
    for modality, path in stage["modalities"].items():
        values = load_directory(path)
        if values is None:
            print(f"{stage['key']} / {modality}: sin embeddings ({path})")
            loaded[(stage["key"], modality)] = None
            continue
        stored = summarize(values, clamp)
        view = values if modality != "SPECT" else diffusion_view(values, calibrate)
        seen = stored if view is values else summarize(view, clamp)
        loaded[(stage["key"], modality)] = {
            "stored": values,
            "view": view,
            "clamp": clamp,
            "title": stage["title"],
        }
        rows.append({
            "etapa": stage["key"],
            "modalidad": modality,
            "vista": "almacenado",
            "clamp": clamp,
            "n_archivos": len(latent_files(path)),
            **stored,
        })
        if modality == "SPECT" and calibrate:
            rows.append({
                "etapa": stage["key"],
                "modalidad": modality,
                "vista": "difusion",
                "clamp": clamp,
                "n_archivos": len(latent_files(path)),
                **seen,
            })
        print(
            f"{stage['key']} / {modality}: {len(latent_files(path))} archivos, "
            f"media {stored['mean']:.3f}, std {stored['std']:.3f}, "
            f"var {stored['variance']:.3f}, |z|>4 {stored['frac_|z|>4']:.4f}"
        )

distribution_table = pd.DataFrame(rows)
show_columns = [
    "etapa", "modalidad", "vista", "n_archivos", "n_voxels",
    "mean", "std", "variance", "min", "max",
    "p0.1", "p1", "p50", "p99", "p99.9",
    "frac_|z|>1", "frac_|z|>4", "frac_fuera_clamp", "clamp",
]
display(
    distribution_table[show_columns].style.format(
        {
            column: "{:.4f}"
            for column in show_columns
            if column not in {"etapa", "modalidad", "vista", "n_archivos", "n_voxels"}
        },
        na_rep="—",
    ).set_caption("Distribución del código. La varianza de la vista difusión SPECT es el MSE de predecir la media.")
)

print("Piso del predictor de la media (varianza de la vista que entrena la difusión SPECT):")
for stage in EMBEDDING_STAGES:
    block = distribution_table[
        (distribution_table["etapa"] == stage["key"]) & (distribution_table["modalidad"] == "SPECT")
    ]
    if block.empty:
        print(f"  {stage['key']}: sin embeddings SPECT")
        continue
    chosen = block[block["vista"] == "difusion"]
    if chosen.empty:
        chosen = block
    variance = float(chosen.iloc[0]["variance"])
    print(f"  {stage['key']}: {variance:.3f}  (clamp ±{stage['clamp']:g})")

In [ ]:
def density_hist(ax, values, color, label):
    """Densidad en la ventana visible. La masa fuera de la ventana no se reescala."""
    sample = sample_values(values, PLOT_SAMPLE)
    counts, edges = np.histogram(sample, bins=80, range=HIST_RANGE)
    widths = np.diff(edges)
    density = counts / (sample.size * widths)
    ax.stairs(density, edges, color=color, linewidth=1.8, label=label)


def plot_overlay(modality: str, field: str, ylabel: str):
    fig, ax = plt.subplots(figsize=(8.5, 4.2))
    drew = False
    xs = np.linspace(HIST_RANGE[0], HIST_RANGE[1], 500)
    for stage in EMBEDDING_STAGES:
        payload = loaded.get((stage["key"], modality))
        if payload is None:
            continue
        density_hist(
            ax,
            payload[field],
            STAGE_COLORS.get(stage["key"], "#4C78A8"),
            stage["title"],
        )
        color = STAGE_COLORS.get(stage["key"], "0.4")
        ax.axvline(payload["clamp"], color=color, linestyle=":", linewidth=1)
        ax.axvline(-payload["clamp"], color=color, linestyle=":", linewidth=1)
        drew = True
    if not drew:
        ax.text(0.5, 0.5, "sin embeddings", ha="center", va="center", transform=ax.transAxes)
    else:
        ax.plot(xs, norm.pdf(xs), color="black", linestyle="--", linewidth=1.2, label="N(0, 1)")
    ax.set_xlim(*HIST_RANGE)
    ax.set_xlabel("valor del vóxel")
    ax.set_ylabel(ylabel)
    ax.set_title(f"{modality}: {ylabel}")
    ax.legend(frameon=False)
    fig.tight_layout()
    plt.show()


plot_overlay("SPECT", "stored", "código almacenado")
plot_overlay("SPECT", "view", "vista difusión")
plot_overlay("MRI", "stored", "código almacenado")

In [ ]:
def qq_points(values: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    sample = np.sort(sample_values(values, QQ_SAMPLE))
    probs = (np.arange(1, sample.size + 1) - 0.5) / sample.size
    return norm.ppf(probs), sample


def panel(ax, values: np.ndarray | None, clamp: float, title: str, color: str, kind: str):
    ax.set_title(title)
    if values is None:
        ax.text(0.5, 0.5, "sin embeddings", ha="center", va="center", transform=ax.transAxes)
        ax.set_xticks([])
        ax.set_yticks([])
        return
    if kind == "hist":
        sample = sample_values(values, PLOT_SAMPLE)
        counts, edges = np.histogram(sample, bins=80, range=HIST_RANGE)
        widths = np.diff(edges)
        ax.stairs(counts / (sample.size * widths), edges, fill=True, color=color, alpha=0.75)
        xs = np.linspace(*HIST_RANGE, 400)
        ax.plot(xs, norm.pdf(xs), color="black", linestyle="--", linewidth=1)
        ax.axvline(clamp, color="#E45756", linestyle=":", linewidth=1)
        ax.axvline(-clamp, color="#E45756", linestyle=":", linewidth=1)
        ax.set_xlim(*HIST_RANGE)
        ax.set_xlabel("valor")
    else:
        theoretical, observed = qq_points(values)
        ax.plot(theoretical, observed, ".", color=color, markersize=2, alpha=0.5)
        limit = float(np.percentile(np.abs(theoretical), 99.5))
        ax.plot([-limit, limit], [-limit, limit], color="black", linestyle="--", linewidth=1)
        ax.set_xlabel("cuantil N(0, 1)")
        ax.set_ylabel("cuantil observado")


modalities = ["SPECT", "MRI"]
fig, axes = plt.subplots(len(EMBEDDING_STAGES), 2, figsize=(10, 3.4 * len(EMBEDDING_STAGES)), squeeze=False)
for row, stage in enumerate(EMBEDDING_STAGES):
    for col, modality in enumerate(modalities):
        payload = loaded.get((stage["key"], modality))
        values = None if payload is None else payload["view" if modality == "SPECT" else "stored"]
        clamp = float(stage["clamp"]) if payload is None else payload["clamp"]
        panel(
            axes[row, col],
            values,
            clamp,
            f"{stage['title']} · {modality}",
            STAGE_COLORS.get(stage["key"], "#4C78A8"),
            "hist",
        )
fig.suptitle("Cada etapa, SPECT (vista difusión) y MRI (almacenado)", y=1.01)
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(len(EMBEDDING_STAGES), 2, figsize=(10, 3.4 * len(EMBEDDING_STAGES)), squeeze=False)
for row, stage in enumerate(EMBEDDING_STAGES):
    for col, modality in enumerate(modalities):
        payload = loaded.get((stage["key"], modality))
        values = None if payload is None else payload["view" if modality == "SPECT" else "stored"]
        panel(
            axes[row, col],
            values,
            float(stage["clamp"]),
            f"QQ {stage['title']} · {modality}",
            STAGE_COLORS.get(stage["key"], "#4C78A8"),
            "qq",
        )
fig.tight_layout()
plt.show()

## 12. Resumen de clasificación E1 (artefactos `embedding_analysis/`)

Lee los `classification_metrics.csv` generados por `analyze_spect_embeddings.py` y `run_phase1_mri_experiment.py` (experimento ROI u otros) y muestra una tabla ordenada por rendimiento, similar al reporte del pipeline automatizado.

In [ ]:
from IPython.display import display
import pandas as pd


E1_CLASSIFICATION_METRICS = {
    "SPECT ROI (E1)": PROJECT_ROOT / "result/exp_nm_s1_spect_rois/embedding_analysis/classification/classification_metrics.csv",
    "MRI ROI (E1)": PROJECT_ROOT / "result/exp_nm_s1_mri_rois/phase1_mri/embedding_analysis/classification/classification_metrics.csv",
    "Synthetic SPECT (E2)": PROJECT_ROOT / "/workspace/IL-CLDM/result/exp_nm_s2_frozen_calib_roi3ch_uni_rois/inference/calib_roi3ch_uni_rois/classification/classification_metrics.csv",
    # Full-brain (descomenta si aplica):
    # "SPECT full-brain": PROJECT_ROOT / "result/exp_nm_s1_spect/embedding_analysis/classification/classification_metrics.csv",
    # "MRI full-brain": PROJECT_ROOT / "result/exp_nm_s1_mri/phase1_mri/embedding_analysis/classification/classification_metrics.csv",
}

METRIC_COLUMNS = ["model", "accuracy", "balanced_accuracy", "f1", "roc_auc"]
SORT_BY = "accuracy"  # o "balanced_accuracy", "f1", "roc_auc"


def load_classification_metrics(path: Path) -> pd.DataFrame:
    if not path.is_file():
        raise FileNotFoundError(f"No existe: {path}")
    df = pd.read_csv(path)
    missing = [c for c in METRIC_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Faltan columnas {missing} en {path}")
    out = df[METRIC_COLUMNS].copy()
    for col in METRIC_COLUMNS[1:]:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    return out.sort_values(SORT_BY, ascending=False).reset_index(drop=True)


def show_metrics_summary(label: str, path: Path) -> pd.DataFrame | None:
    print(f"### {label}")
    print(path)
    if not path.is_file():
        print("  (archivo no encontrado — omite o ajusta la ruta)\n")
        return None
    ranked = load_classification_metrics(path)
    styled = (
        ranked.style.format(
            {c: "{:.6f}" for c in METRIC_COLUMNS[1:]},
            na_rep="—",
        )
        .background_gradient(subset=["accuracy", "balanced_accuracy", "f1", "roc_auc"], cmap="Greens")
        .set_caption(f"Clasificadores ordenados por {SORT_BY}")
    )
    display(styled)
    best = ranked.iloc[0]
    print(
        f"Mejor ({SORT_BY}): {best['model']} — "
        f"acc={best['accuracy']:.3f}, bal_acc={best['balanced_accuracy']:.3f}, "
        f"f1={best['f1']:.3f}, AUC={best['roc_auc']:.3f}\n"
    )
    return ranked


spect_ranked = show_metrics_summary("SPECT ROI (E1)", E1_CLASSIFICATION_METRICS["SPECT ROI (E1)"])
mri_ranked = show_metrics_summary("MRI ROI (E1)", E1_CLASSIFICATION_METRICS["MRI ROI (E1)"])
synthetic_ranked = show_metrics_summary("Synthetic SPECT (E2)", E1_CLASSIFICATION_METRICS["Synthetic SPECT (E2)"])

In [ ]:
# Tabla comparativa SPECT vs MRI (mejor visión global)
frames = []
for label, path in E1_CLASSIFICATION_METRICS.items():
    if not path.is_file():
        continue
    block = load_classification_metrics(path)
    block.insert(0, "modality", label)
    frames.append(block)

if frames:
    comparison = pd.concat(frames, ignore_index=True)
    pivot = comparison.pivot_table(
        index="model",
        columns="modality",
        values=SORT_BY,
        aggfunc="first",
    )
    print(f"Comparación de {SORT_BY} por modelo y modalidad:")
    display(pivot.style.format("{:.4f}", na_rep="—").background_gradient(cmap="Greens", axis=None))

    fig, ax = plt.subplots(figsize=(8, max(3, 0.45 * len(comparison))))
    y_labels = comparison["modality"] + " / " + comparison["model"]
    x_vals = comparison[SORT_BY].values
    colors = ["#4C78A8" if "SPECT" in m else "#F58518" for m in comparison["modality"]]
    ax.barh(y_labels, x_vals, color=colors)
    ax.set_xlabel(SORT_BY)
    ax.set_title(f"Desempeño E1 por clasificador ({SORT_BY})")
    ax.set_xlim(0, 1.05)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("No hay classification_metrics.csv en las rutas configuradas.")